Real time traffic violation detection system 

Vehicle collection

Description: This collection stores all registered vehicles in the AWAS system. This allows the system to identify vehicle ownership, associate license plates with owner information, and register violations consistently over time. Storing this data centrally ensures that any flagged violation can be traced back to the vehicle's owner for enforcement and legal purposes.   

Document schema:
{
   "car_plate": "string",
   "owner_name": "string",
   "owner_addr": "string",
   "vehicle_type": "string",
   "registration_date": "ISODate"
   
}
   
 
Sample document:  
{
  "car_plate": "WXY1234",
  "owner_name": "Ahmad bin Salleh",
  "owner_addr": "12, Jalan Damai, Kuala Lumpur",
  "vehicle_type": "Perodua Myvi",
  "registration_date": ISODate("2020-08-14T10:32:00Z")
}

Indexes: 
    Field: car_plate
    Type: Single Field Index
    Purpose: Unique car plate number allows for fast lookups when querying for a vehicle using car plate

Shard Key Strategy:
    Chosen shard key: car_plate
    Shard key type: Hashed Based sharding
    Rationale: Since car plates are unique and uniformly distributed, by using a hashed shard key on car_plate, data is evenly
    distributed across shards, which ensures horizontal scalability for high throuput operations like lookups and updates.
    
Data Retention Policy: Retain indefinitely unless vehicle is officially deregistered by officials. This is because vehicle registration data is critical for tracking ownership and linking violations to responsible parties. 


Camera collection

Description: This collection stores all fixed AWAS camera metadata, including the camera position along the highway, latlong coordinates and speed limit.

Document schema:
{
   "camera_id": "int",
   "latitude": "float",
   "longtitude": "float",
   "position:": "float",
   "speed_limit": "int"

}

Sample document:

{
  "camera_id": 101,
  "latitude": 3.1390,
  "longitude": 101.6869,
  "position": 15.5,
  "speed_limit": 110.0
}

Indexes:
    Field: camera_id
    Type: Single Field Index
    Purpose: For quick lookup of camera information using the unique camera id 
    
Shard Key Strategy:
    Chosen shard key: camera_id
    Shared key type: Hashed based sharding
    Rationale: Since camera id is unique and uniformly distributed, hashed sharding ensures even data distribution across 
    shards. Note here I did consider range based sharding, however since the camera id is uniformally distributed, this could 
    cause uneven load if many many queries hit the same key range.
    
Data Retention Policy: The camera collection is stored permanently to ensure accurate and consistent historical analysis and enforcement traceability.

Violation collection

Description: This collection stores all violations detected by the AWAS system. This enables enforcement authorities to track and issue penalties for average-speed and instantaneous-speed violations on Malaysian roads.

Document schema:
{
   "car_plate": "string",
   "violation_date": "string",
   "violation_id": "UUID",
   "violations": [
        {
         "timestamp": "string",
         "camera_id": "int",
         "speed_read": "double",
         "speed_limit": "double",
         "violation_type": "string",
         "time_added": "string"
       },
       {
       "start_timestamp": "string",
       "end_timestamp": "string",
       "camera_1": "int",
       "camera_2: "int",
       "average_speed": "double",
       "speed_limit": "double",
       "violation_type": "string",
       "time_added": "string"
       
       
       }
   ]
}

Sample document:

{
  "car_plate": "WXD1234",
  "violation_date": "2025-05-28",
  "violation_id": "V-7c24b85e-b1d0-4f30-b5c1-9a3e47de8122",
  "violations": [
    {
      "timestamp": "2025-05-28T08:15:23",
      "camera_id": 101,
      "speed_read": 112.5,
      "speed_limit": 90.0,
      "violation_type": "instantaneous speed_violation",
      "time_added": "2025-05-28T08:16:00"
    },
    {
      "start_timestamp": "2025-05-28T10:00:00",
      "end_timestamp": "2025-05-28T10:02:30",
      "camera_1": 102,
      "camera_2": 103,
      "average_speed": 118.0,
      "speed_limit": 90.0,
      "violation_type": "average speed_violation",
      "time_added": "2025-05-28T10:03:00"
    }
  ]
}


Indexes:
    Field: car_plate, violation_date
    Type: Compound field index
    Purpose: To optimize queries for retrieving all violations for a specific car on a given day. Also prevents duplication and 
    ensures uniqueness.
    
Shard Key Startegy:
    Chosen shard key: violation_id
    Shard key type: Hashed based sharding
    Rationale: Violdation id is unique, ensuring even distribution across shards. 
    
    
Data Retention Policy: Violations are retained for at least 10 years, for regulatory and legal audit requirements. Data older 
than 10 years may be archived or purged, depending on the enforcement policy.
    
  


Relationship between Vehicle collection and Violation collection: One to Many Relationship

For each vehicle in registered in the vehicle collection, they may have 0 or more violations.
Modelling choice: 
For each violation document in the collection, it will contain a reference by car plate to the vehicle collection.

Justification:

Read/Write patterns:
Violations are frequently updated and queried independently of vehicle metadata. In this case, we use referencing to avoid rewriting the entire document with each new violation.

Data Duplication vs Join Cost:
Embedding vehicle metadata would lead to excessive redundancy, especially for vehicles with many violations. This would make updates(such as changing vehicle owners) complex and error-prone. Referencing avoids duplication and ensures consistency, as vehicle details are stored and maintained in one place. The cost of joins is acceptable since we rarely need full vehicle metadata.

Consistency Requirements:
Since vehicle information does not change often, and car plates never change, referencing the car plate ensures that all violations point to the correct vehicle, even if vehicle metadata changes.This model eliminates the risk of inconsistencies caused by outdated embedded data across many documents.



Relationship between Violation collection and Camera collection: One to Many Relationship

For each camera in the collection, they may capture many violations.
Modelling choice:
For each violation entry within a violation document, it will contain a reference by camera id that captured the speed violation.

Justification:
Read/Write Patterns:
Camera metadata is rarely updated, while violations are continuously written, and queried. And since most queries would only be for camera id, and not other camera metadata, we can use referencing camera id to avoid bloating violation documents and also support efficient, high-speed writes and reads.

Data Duplication vs Join Cost
Embedding static camera metadata in every violation would waste storage and complicate updates if camera information changes. Referencing via camera_id avoids duplication and simplifies maintenance. The cost of joins is acceptable since full camera metadata is rarely required.

Consistency Requirements:
Since camera information does not change often(in most cases not at all), referencing camera via camera id ensures that all violations point to the correct camera that captured it. This design maintains data consistency, as any updates to camera details only need to be made in one place—eliminating the risk of inconsistencies across multiple violation documents.





Consistency and Idempotency

Idempotent Writes:
Yes, the model supports idempotent writes. In the violation collection, each document is uniquely identified by the combination of car_plate and violation_date, while individual violations within the violations array include a unique violation_id (UUID). This ensures that the same violation is not inserted multiple times, even if the same write is retried due to a transient failure.

Upsert Pattern in Violation Collection:

An upsert pattern is used when inserting new violations:

If a document with the same car_plate and violation_date exists, the system performs an update by pushing the new violation into the existing violations array (using $push). If no such document exists, a new document is inserted. This ensures both data integrity and support for idempotency.

Scalability and Fault-Tolerance
High Ingest Rates:
The data model supports high ingest rates by using hashed sharding on the violation_id, which distributes write load evenly across shards. Also, by using compound indexing on (car_plate, violation_date), we can do efficient upsert operations without full document scans.

Low-Latency Lookups:
The model supports low-latency lookups by creating a compound index on (car_plate, violation_date) for quick access to violations by a vehicle on a specific day. Also, by keeping camera and vehicle metadata separate in their respective collections and referring via IDs (camera_id, car_plate), the violation documents are smaller and reduces retrieval time for high-frequency queries that don’t require full metadata.

Trade-Offs:
Using references (instead of embedding) avoids redundancy and allows consistent updates to vehicle and camera metadata. However, there are occasional join costs when full metadata is needed.


In [1]:
# my imports

import json
import pymongo
import uuid
from pymongo import MongoClient, UpdateOne
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import expr
from pyspark.sql.streaming import DataStreamWriter
from datetime import datetime
import pandas as pd


In [2]:
def insert_vehicle_data(db, vehicle_csv_path):
    """
    Loads vehicle data from a CSV file, converts appropriate fields,
    and inserts the data into the 'vehicles' collection in MongoDB.

    """
    vehicle_df = pd.read_csv(vehicle_csv_path)

    # convert registration_date to datetime
    vehicle_df['registration_date'] = pd.to_datetime(vehicle_df['registration_date'], errors='coerce')
    
    # insert to mongodb
    db['vehicles'].insert_many(vehicle_df.to_dict('records'))
    print(f"Inserted {len(vehicle_df)} new records into 'vehicle' collection.")

def insert_camera_data(db, camera_csv_path):
    """
    Loads camera metadata from a CSV file, converts appropriate fields,
    and inserts the data into the 'cameras' collection in MongoDB.

    """
    camera_df = pd.read_csv(camera_csv_path)

    # Cast fields to appropriate types
    camera_df['camera_id'] = pd.to_numeric(camera_df['camera_id'], errors='coerce').astype('Int64')
    camera_df['speed_limit'] = pd.to_numeric(camera_df['speed_limit'], errors='coerce').astype('Int64')
    camera_df['latitude'] = pd.to_numeric(camera_df['latitude'], errors='coerce')
    camera_df['longitude'] = pd.to_numeric(camera_df['longitude'], errors='coerce')

   

    # insert to mongodb
    db['cameras'].insert_many(camera_df.to_dict('records'))
    print(f"Inserted {len(camera_df)} new records into 'camera' collection.")
    
    


host_ip = "10.192.101.151"
db_name = "fit3182_a2_db_2"

# File paths
vehicle_csv = "vehicle.csv"
camera_csv = "camera.csv"

# Connect to MongoDB
client = MongoClient( host=f'{host_ip}',
        port=27017)
db = client[db_name]

# Insert data
insert_vehicle_data(db, vehicle_csv)
insert_camera_data(db, camera_csv)

client.close()



Inserted 10000 new records into 'vehicle' collection.
Inserted 3 new records into 'camera' collection.


In [3]:

import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

# kafka broker ip adrress
host_ip = "10.192.101.151"

# Initialize SparkSession with local master
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('Camera Event Data')
    .getOrCreate()
)



In [4]:
def read_kafka_topic(topic):
    """
    Read a Kafka topic as a streaming DataFrame.
    """
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"{host_ip}:9092")
        .option("subscribe", topic)
        .load()
    )


In [5]:

# define schema for parsing kafkfa json values
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("batch_id", IntegerType(), True),
    StructField("car_plate", StringType(), True ),
    StructField("camera_id", IntegerType(), True),
    StructField("timestamp", StringType(), True),
    StructField("speed_reading", DoubleType(), True)
])

# load static camera metadata from csv
camera_df = spark.read.csv("camera.csv", header=True)
 
# cast types to column and drop uneccessary columns
camera_df = camera_df \
    .withColumn("camera_id", col("camera_id").cast("int")) \
    .withColumn("speed_limit", col("speed_limit").cast("double")) \
    .withColumn("latitude", col("latitude").cast("double")) \
    .withColumn("longitude", col("longitude").cast("double")) \
    .withColumn("position", col("position").cast("double")) \
    .drop("latitude", "longitude")



def consumer(topic, prefix):
    """
    Process Kafka streaming data by parsing JSON and joining with static camera data.

    """
    read_topic = read_kafka_topic(topic)
    
    # Parse JSON from Kafka and convert to structured DataFrame
    df = (
        read_topic
        .select(from_json(col("value").cast("string"), schema).alias("data"))
        .select("data.*")
        .withColumn("timestamp", to_timestamp(col("timestamp")))  
        .drop("event_id") 
        .alias(prefix)
        .join(broadcast(camera_df.alias(f"{prefix}_extra")), col(f"{prefix}.camera_id") == col(f"{prefix}_extra.camera_id"))
        # broadcast the static camera metadata
    )
  
    return df
    

# Create streaming DataFrames for each Kafka topic

df_a = consumer("camera_event_171", "camera_a")
df_b = consumer("camera_event_172", "camera_b")
df_c = consumer("camera_event_173", "camera_c")

In [6]:
def instant_violation(df, prefix):
    """
    Detect instantaneous speed violations where a vehicle's speed exceeds the limit at a single camera.

    """
    return (
        df.filter(col(f"{prefix}.speed_reading") > col(f"{prefix}_extra.speed_limit")) # filter for speeds higher than speed limit
          .select(
              
              col(f"{prefix}.car_plate"),
              col(f"{prefix}.camera_id").alias("camera_1_id"), # keep this for union, need same size columns
              col(f"{prefix}.camera_id").alias("camera_2_id"),
              col(f"{prefix}.timestamp").alias("time_start"),
              col(f"{prefix}.timestamp").alias("time_end"),
              col(f"{prefix}.speed_reading").alias("speed_reading"),
              col(f"{prefix}_extra.speed_limit").alias("speed_limit"),
              lit("instantaneous_speed_violation").alias("violation_type") # add violation type
          )
    )
    
# filter each stream for instant violations   
    
df_a_violation = instant_violation(df_a, "camera_a")

df_b_violation = instant_violation(df_b, "camera_b")

df_c_violation = instant_violation(df_c, "camera_c")

In [7]:

def non_instant_violation(df, prefix):
    """
    Extract non-violating vehicle entries (speeds within limit) from a single camera.
    This is for logging non violating vehicles

    """
    return (
        df.filter(col(f"{prefix}.speed_reading") <= col(f"{prefix}_extra.speed_limit"))
          .select(
              col(f"{prefix}.car_plate"),
              col(f"{prefix}.camera_id").alias("camera_1_id"),
              col(f"{prefix}.camera_id").alias("camera_2_id"),
              col(f"{prefix}.timestamp").alias("time_start"),
              col(f"{prefix}.timestamp").alias("time_end"),
              col(f"{prefix}.speed_reading").alias("speed_reading"),
              col(f"{prefix}_extra.speed_limit").alias("speed_limit"),
              lit("non_instantaneous_speed_violation").alias("violation_type")
          )
    )


# filter each stream for non instant violations

df_a_non_violation = non_instant_violation(df_a, "camera_a")

df_b_non_violation = non_instant_violation(df_b, "camera_b")

df_c_non_violation = non_instant_violation(df_c, "camera_c")
    

In [8]:
# here i do my joins using watermarking with time of 10 minutes and matching car plates and window of 5 minutes

df_ab = (
    df_a.withWatermark("timestamp", "10 minutes")
        .join(
            df_b.withWatermark("timestamp", "10 minutes"),
            expr("""
                camera_a.car_plate = camera_b.car_plate AND
                camera_a.timestamp BETWEEN camera_b.timestamp - interval 5 minutes AND camera_b.timestamp + interval 5 minutes
            """)
        )
)


df_bc = (
    df_b.withWatermark("timestamp", "10 minutes")
        .join(
            df_c.withWatermark("timestamp", "10 minutes"),
            expr("""
                camera_b.car_plate = camera_c.car_plate AND
                camera_b.timestamp BETWEEN camera_c.timestamp - interval 5 minutes AND camera_c.timestamp + interval 5 minutes
            """)
        )
)





In [9]:

# I add an extra column for calculating my average speed within the 2 cameras in my joined stream


# calculate average speed between camera 1 and camera 2 
df_ab_avg_speed = df_ab.withColumn(
    "avg_speed_kmh",  
    round(
        (col("camera_b_extra.position") - col("camera_a_extra.position")) * 3600 /
        (col("camera_b.timestamp").cast("double") - col("camera_a.timestamp").cast("double")),
        1
    )
        
    
)


# calculate average speed between camera 2 and camera 3 
df_bc_avg_speed = df_bc.withColumn(
    "avg_speed_kmh",
    round(
        (col("camera_c_extra.position") - col("camera_b_extra.position")) * 3600 /
        (col("camera_c.timestamp").cast("double") - col("camera_b.timestamp").cast("double")),
        1
    )
)


def avg_violation(df, prefix_1, prefix_2):
    """
    This function is to filter for average speed violations 
    """
    return (df.filter((col("avg_speed_kmh")> col("camera_b_extra.speed_limit")))  # this is filtering step
           .select(
           col(f"{prefix_1}.car_plate"),  
           col(f"{prefix_1}.camera_id").alias("camera_1_id"),
           col(f"{prefix_2}.camera_id").alias("camera_2_id"),
           col(f"{prefix_1}.timestamp").alias("time_start"),
           col(f"{prefix_2}.timestamp").alias("time_end"),
           col("avg_speed_kmh").alias("speed_reading"),
           col(f"{prefix_2}_extra.speed_limit").alias("speed_limit"),
           lit("average_speed_violation").alias("violation_type")))  # add violation type



# now i filter for both my streams
df_avg_ab_violation = avg_violation(df_ab_avg_speed, "camera_a", "camera_b")
df_avg_bc_violation = avg_violation(df_bc_avg_speed, "camera_b", "camera_c")





In [10]:

# I split my streams into 2 streams since if i union all 5 streams together, the batch gets too big

# I union the streams with instant speed violations(at camera 1,2,3)
instantaneous_violations = df_a_violation.union(df_b_violation).union(df_c_violation)

# I union the streams with average speed violations(between camera 1&2, camera 2&3)
average_speed_violations = df_avg_ab_violation.union(df_avg_bc_violation)


# I union the streams with non instant speed violations(for logging)
non_instant_violations = df_a_non_violation.union(df_b_non_violation).union(df_c_non_violation)



In [11]:

# write to stream
log_to_console = (
    non_instant_violations.writeStream.outputMode('append').format('console')
)

In [12]:

def to_iso(dt):
    """
    This function converts a datetime object to ISO 8601 formatted string (YYYY-MM-DDTHH:MM:SS).

    """

    return dt.strftime("%Y-%m-%dT%H:%M:%S")

def process(batch_df, batch_id):
    """
    Process each micro-batch of violations from the streaming DataFrame, and write them into MongoDB.

    For each violation record in the batch: I first convert timestamps to ISO format strings.
    Then, I create or update(if it exists) a MongoDB document keyed by car_plate and violation_date.
    I append new violation details into the violations array in the document.
    I use bulk write function here for effiency, add indexes for effient lookup, with idempotent writes.
    
    """
    rows = batch_df.collect()
    if not rows:
        return

    client = MongoClient( host=f'{host_ip}',  # connect to my mongo
            port=27017)  
    db = client["fit3182_a2_db_1"]
    collection = db["violations"]

    # create my indexes
    collection.create_index("violation_id", unique=True)
    collection.create_index([("car_plate", 1), ("violation_date", 1)])


    operations = []

    # create violation record depending on violation type
    for row in rows:
        car_plate = row["car_plate"]
        violation_type = row["violation_type"]  
        v = row  

        if violation_type == "instantaneous_speed_violation":
            record = {
                "timestamp": to_iso(v["time_start"]),
                "camera_id": v["camera_1_id"],
                "violated_speed": v["speed_reading"],
                "speed_limit": v["speed_limit"],
                "violation_type": v["violation_type"],
                "time_added": datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
            }
        else:
            record = {
                "start_timestamp": to_iso(v["time_start"]),
                "end_timestamp": to_iso(v["time_end"]),
                "camera_1": v["camera_1_id"],
                "camera_2": v["camera_2_id"],
                "violated_speed": v["speed_reading"],
                "speed_limit": v["speed_limit"],
                "violation_type": v["violation_type"],
                "time_added": datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
            }

        # add violation date
        violation_date = v["time_start"].date().strftime("%Y-%m-%d")  

        violation_record = {
            
            "violation_id": "V-" + str(uuid.uuid4()),   # unique violation id 
            "car_plate": car_plate,
            "violation_date": violation_date,
            "violations": [record]
        }
        
        # add bulk upsert operation  into a list to insert or update violation record per car_plate & date

        update = UpdateOne(
            {
                "car_plate": car_plate,
                "violation_date": violation_date
            },
            {
                "$setOnInsert": {
                   
                    "violation_id": violation_record["violation_id"],
                    "car_plate": car_plate,
                    "violation_date": violation_date
                },
                "$push": {
                    "violations": {
                        "$each": violation_record["violations"]
                    }
                }
            },
            upsert=True  # for idempotency
        )

        operations.append(update)
        
    # Execute bulk write if there are any operations
        
    if operations:
        result = collection.bulk_write(operations)
        print(f"[BATCH {batch_id}] Inserted: {result.upserted_count}, Modified: {result.modified_count}")


    client.close()
    
# write to stream(i have 2 streams so write twice)
writer_1 = (
    instantaneous_violations.writeStream
        .foreachBatch(process)    
        .outputMode("append")
        .option("checkpointLocation", "./checkpoints/violation")
        
)

writer_2 = (
    average_speed_violations.writeStream
        .foreachBatch(process)    
        .outputMode("append")
        .option("checkpointLocation", "./checkpoints/violation")
)



In [13]:


try:
    # start the streaming queries for instantaneous speed violations
    query_1 = writer_1.start()
    # start the streaming queries for average speed violations
    query_2 = writer_2.start()
    # start the streaming query that logs non violating vehicles to console 
    query_3 = log_to_console.start()
    query_1.awaitTermination()
    query_2.awaitTermination()
    query_3.awaitTermination()
    
    
except KeyboardInterrupt:
    print('Interrupted by CTRL-C. Stopping query.')
finally:
    # stop all streaming queries
    query_1.stop()
    query_2.stop()
    query_3.stop()

[BATCH 1] Inserted: 7, Modified: 0
[BATCH 2] Inserted: 3, Modified: 0
[BATCH 3] Inserted: 91, Modified: 3
[BATCH 4] Inserted: 2, Modified: 0


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.8/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/conda/lib/python3.8/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/conda/lib/python3.8/socket.py", line 669, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


Interrupted by CTRL-C. Stopping query.


ERROR:py4j.clientserver:There was an exception while executing the Python Proxy on the Python Side.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.8/site-packages/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
  File "/opt/conda/lib/python3.8/site-packages/pyspark/sql/utils.py", line 272, in call
    raise e
  File "/opt/conda/lib/python3.8/site-packages/pyspark/sql/utils.py", line 269, in call
    self.func(DataFrame(jdf, self.session), batch_id)
  File "/tmp/ipykernel_55787/1316453545.py", line 19, in process
    rows = batch_df.collect()
  File "/opt/conda/lib/python3.8/site-packages/pyspark/sql/dataframe.py", line 817, in collect
    sock_info = self._jdf.collectToPython()
  File "/opt/conda/lib/python3.8/site-packages/py4j/java_gateway.py", line 1321, in __call__
    return_value = get_return_value(
  File "/opt/conda/lib/python3.8/site-packages/pyspark/sql/utils.py", line 190, in deco
    return f(